# Inner-iteration time: CG vs CBCD (Water vs MeOH)

Absolute time spent in the inner iterations (ms, Fock builds excluded; measured in C by `fast_scf`, median of repeated runs, 6-31++G**) for CG and CBCD ($c = N_o$). All bars have equal width, so bar area is proportional to inner-iteration time. Dashed outline: estimated noninteracting supersystem.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
from pyscf import gto, scf

# Locate rhf/code_in_c from the repo root (the dir containing manuscript/), so this works regardless of
# where the kernel was started.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ref_data").is_dir())
sys.path.insert(0, str(ROOT / "pyscf_methods" / "code_in_c"))
from fast_scf import cg_scf, cbcd_scf        # drop-ins for rhf/cg_scf.py, rhf/cbcd_scf.py (whole SCF in C)
from oracles import build_integrals, make_rhf_oracle_energy, make_rhf_grad_energy   # fast_scf puts ground_state/ on sys.path

mol1 = "Water"
mol2 = "MeOH"
ref_name = {mol1: "H2O", mol2: "CH3OH"}      # ref_data/<name>.txt
BASIS, REPS, EPS, DAMP = "6-31++G**", 101, 1e-6, 0.4

systems = {}
for mol in (mol1, mol2):
    with open(ROOT / "ref_data" / f"{ref_name[mol]}.txt") as fh:
        atom = "".join(fh.readlines()[1:])               # skip header row
    m = gto.M(atom=atom, basis=BASIS, verbose=0)
    ne = m.nelectron
    f = make_rhf_oracle_energy(build_integrals(m), ne)
    g0 = scf.RHF(m); g0.max_cycle = 0; g0.kernel()
    X = np.real(sp.linalg.fractional_matrix_power(g0.get_ovlp(), 0.5))
    systems[mol] = dict(f=f, grad=make_rhf_grad_energy(f), C0=X @ g0.mo_coeff[:, : ne // 2], no=ne // 2,
                        e_ref=scf.RHF(m).kernel() - m.get_enuc())
N_o = {mol: s["no"] for mol, s in systems.items()}

# timing=True returns (C, E, (wall, Fock-build time)); the inner-iteration time is wall - Fock-build time,
# as in pyscf_wrapper.ipynb. Median over REPS runs. CG: cg_scf; CBCD: cbcd_scf with c = N_o (Gauss-Southwell).
# Inner iteration counts come from one opt_profiling run (inner objective per outer iteration, start + each step).
def run(method, s):
    if method == "CG":
        return lambda **kw: cg_scf(s["f"], s["grad"], np.copy(s["C0"]), eps=EPS, damp_param=DAMP, **kw)
    return lambda **kw: cbcd_scf(s["f"], s["grad"], np.copy(s["C0"]), s["no"], eps=EPS, damp_param=DAMP, **kw)

t_inner, wall, iters = {"CG": {}, "CBCD": {}}, {"CG": {}, "CBCD": {}}, {"CG": {}, "CBCD": {}}
for key in ("CG", "CBCD"):
    for mol, s in systems.items():
        call = run(key, s)
        res = [call(timing=True) for _ in range(REPS)]       # (C, E, (wall, t_fock))
        assert all(abs(r[1] - s["e_ref"]) < 1e-4 for r in res), f"{key} {mol}: energy disagrees with PySCF"
        t_inner[key][mol] = float(np.median([r[2][0] - r[2][1] for r in res]))
        wall[key][mol] = float(np.median([r[2][0] for r in res]))
        _, _, (_, subs) = call(opt_profiling=True)
        iters[key][mol] = sum(len(x) - 1 for x in subs)
        print(f"{key:5s} {mol:5s} N_o={N_o[mol]}  inner time = {t_inner[key][mol] * 1e3:8.2f} ms  "
              f"total wall = {wall[key][mol] * 1e3:.2f} ms  inner iterations = {iters[key][mol]}")

CG    Water N_o=5  inner time =     0.13 ms  total wall = 0.51 ms  inner iterations = 74
CG    MeOH  N_o=9  inner time =     0.63 ms  total wall = 5.57 ms  inner iterations = 94
CBCD  Water N_o=5  inner time =     0.07 ms  total wall = 0.43 ms  inner iterations = 217
CBCD  MeOH  N_o=9  inner time =     0.39 ms  total wall = 5.15 ms  inner iterations = 440


In [ ]:
from matplotlib.patches import Polygon, Patch

w = 0.35                      # bar width (equal for every bar, so area is proportional to inner-iteration time)
centers = {"CG": 0.0, "CBCD": 1.2}
colors = {mol1: "#4C72B0", mol2: "#DD8452"}
labels = {mol: rf"{name} ($N_o$ = {N_o[mol]})" for mol, name in [(mol1, mol1), (mol2, mol2)]}
pad_x, pad_y = 0.0, 0.0       # outline follows bar edges exactly
dash_kw = dict(fill=False, edgecolor="black", linestyle=(0, (3, 1.5)), linewidth=2.6, zorder=3)
dash_label = r"Noninteracting Supersystem (estimated)"

ms = {m: {mol: t * 1e3 for mol, t in d.items()} for m, d in t_inner.items()}   # absolute inner-iteration time in ms

fig, ax = plt.subplots(figsize=(5.5, 4.2))
for method, xc in centers.items():
    h = ms[method]                    # bar heights for this group: absolute inner-iteration time (ms)
    left = xc - w                     # center the flush pair on xc
    for mol, x0 in [(mol1, left), (mol2, left + w)]:
        y = h[mol]
        ax.bar(x0, y, width=w, align="edge", color=colors[mol],
               edgecolor="black", linewidth=0.8,
               label=labels[mol] if method == "CG" else None)
        ax.text(x0 + w / 2, y + 0.01 * max(max(d.values()) for d in ms.values()), f"{y:.2f} ms",
                ha="center", va="bottom", fontsize=9, linespacing=0.95)

    # Dashed supersystem outline
    xl, xm, xr = left - pad_x, left + w, left + 2 * w + pad_x
    y1, y2 = h[mol1] + pad_y, h[mol2] + pad_y
    if method == "CG":        # single box at the taller (MeOH) height
        verts = [(xl, 0), (xl, y2), (xr, y2), (xr, 0)]
    else:                     # step outline hugging each bar's height
        verts = [(xl, 0), (xl, y1), (xm, y1), (xm, y2), (xr, y2), (xr, 0)]
    ax.add_patch(Polygon(verts, closed=False, **dash_kw))

ax.set_xticks(list(centers.values()))
tick_labels = {"CG": "CG", "CBCD": r"CBCD, $c = N_o$"}
ax.set_xticklabels([tick_labels[m] for m in centers])
ax.set_ylabel("Inner-iteration time (ms)")
ax.set_ylim(0, max(max(d.values()) for d in ms.values()) * 1.12)
ax.spines[["top", "right"]].set_visible(False)
handles, lbls = ax.get_legend_handles_labels()
handles.append(Patch(**{k: v for k, v in dash_kw.items() if k != "zorder"}))
lbls.append(dash_label)
ax.legend(handles, lbls, frameon=False, fontsize=9, ncol=2,
          loc="lower center", bbox_to_anchor=(0.5, 1.0),
          handlelength=1.6, handleheight=0.9, handletextpad=0.5,
          labelspacing=0.35, columnspacing=1.2, borderaxespad=0.2)
fig.tight_layout()
fig.savefig("rel_time_bar.pdf", bbox_inches="tight")
fig.savefig("rel_time_bar.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from pathlib import Path

# Walk up from the notebook's cwd to the repo root (the dir containing manuscript/),
# so this works regardless of where the kernel was started.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ref_data").is_dir())
FIGDIR = ROOT.parent / "manuscript" / "new_figures"

fig.savefig(FIGDIR / "rel_time_bar.pdf", bbox_inches="tight")
fig.savefig(FIGDIR / "rel_time_bar.png", dpi=300, bbox_inches="tight")
print("wrote to", FIGDIR)